In [2]:
import litellm
import json

# A collection of unstructured corporate log entries and specifications
raw_ingestion_pool = [
    "CRITICAL ALERT: Node 04 on the MeshQuery cluster coordination ring dropped heartbeat signals at 14:22 UTC. Port 9092 is completely unreachable.",
    "Our standard engineering onboarding manual mandates that all fresh hires pull down the setup script to establish an isolated local Ubuntu workspace.",
    "MeshQuery architecture leverages spatial hashing clusters mapped via H3 structures to preserve ultra-low latency index lookups.",
    "URGENT: The primary payment database threw an out-of-memory exception during the multi-region validation loop check."
]

print("✓ Cell 1: Ingestion pool populated with 4 distinct corporate strings.")

17:58:20 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
17:58:21 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


✓ Cell 1: Ingestion pool populated with 4 distinct corporate strings.


In [1]:
def enrich_chunk_metadata(raw_text: str) -> dict:
    # Build a strict classification prompt instructing the model to act as a data librarian
    prompt = [
        {
            "role": "system",
            "content": (
                "You are an expert technical data librarian. Your task is to analyze the provided text fragment "
                "and extract specific structural metadata fields.\n\n"
                "You must extract exactly these four keys:\n"
                "1. 'system' (e.g., 'MeshQuery', 'Database', 'Onboarding', or 'Unknown')\n"
                "2. 'category' (e.g., 'Infrastructure', 'Incident_Report', 'Documentation')\n"
                "3. 'severity' (e.g., 'Critical', 'Urgent', 'Routine')\n"
                "4. 'tags' (A list of up to 3 short lowercase keyword string tags)\n\n"
                "Output your response ONLY as a single valid JSON object. Do not include markdown code blocks, explanation text, or preambles."
            )
        },
        {
            "role": "user",
            "content": f"Text to analyze: {raw_text}"
        }
    ]
    
    response = litellm.completion(
        model="ollama/llama3.2",
        messages=prompt,
        api_base="http://localhost:11434",
        temperature=0.0  # Force maximum determinism
    )
    
    raw_content = response.choices[0].message.content.strip()
    
    # Standard cleanup handle in case local models append markdown ticks
    if raw_content.startswith("```json"):
        raw_content = raw_content.replace("```json", "").replace("```", "").strip()
    elif raw_content.startswith("```"):
        raw_content = raw_content.replace("```", "").strip()

    try:
        enriched_metadata = json.loads(raw_content)
    except Exception:
        print(f"⚠️ JSON extraction failed on raw string: {raw_content}")
        # Structured fallback block to protect pipeline execution
        enriched_metadata = {"system": "Unknown", "category": "Unclassified", "severity": "Routine", "tags": []}
        
    return enriched_metadata

print("✓ Cell 2: Metadata enrichment processing loop successfully compiled.")

✓ Cell 2: Metadata enrichment processing loop successfully compiled.


In [3]:
# Create a destination list to act as our production database records
final_document_database = []

print("--- Starting Automated Enrichment Run ---\n")

for idx, text in enumerate(raw_ingestion_pool, 1):
    print(f"Processing Text Chunk #{idx}...")
    
    # Extract the structured attributes via the judge model
    extracted_meta = enrich_chunk_metadata(text)
    
    # Assemble the final database record
    enriched_record = {
        "text_content": text,
        "metadata": extracted_meta
    }
    
    final_document_database.append(enriched_record)
    
    print(f"  System  : {extracted_meta.get('system')}")
    print(f"  Category: {extracted_meta.get('category')}")
    print(f"  Severity: {extracted_meta.get('severity')}")
    print(f"  Tags    : {extracted_meta.get('tags')}\n")

print("✓ Cell 3: Processing complete. All document slices fully enriched with structural indices.")

--- Starting Automated Enrichment Run ---

Processing Text Chunk #1...
  System  : MeshQuery
  Category: Infrastructure
  Severity: Critical
  Tags    : ['meshquery', 'heartbeat', 'unreachable']

Processing Text Chunk #2...
  System  : Onboarding
  Category: Documentation
  Severity: Routine
  Tags    : ['engineering', 'manual', 'onboarding']

Processing Text Chunk #3...
  System  : MeshQuery
  Category: Infrastructure
  Severity: Routine
  Tags    : ['spatial_hashing', 'h3_structures']

Processing Text Chunk #4...
  System  : Database
  Category: Incident_Report
  Severity: Urgent
  Tags    : ['payment', 'database', 'outofmemory']

✓ Cell 3: Processing complete. All document slices fully enriched with structural indices.


In [4]:
# Simulate an operations engineer searching exclusively for critical infrastructure faults
print("==================================================================")
print("     DATABASE DISCOVERY PASS: CRITICAL INFRASTRUCTURE ALERTS      ")
print("==================================================================")

for record in final_document_database:
    meta = record["metadata"]
    
    # Apply a multi-key hard filter pass before executing any textual search
    if meta.get("severity") == "Critical" and meta.get("system") == "MeshQuery":
        print(f"MATCH FOUND!")
        print(f"Content: \"{record['text_content']}\"")
        print(f"Metadata Flags: {meta}\n")

     DATABASE DISCOVERY PASS: CRITICAL INFRASTRUCTURE ALERTS      
MATCH FOUND!
Content: "CRITICAL ALERT: Node 04 on the MeshQuery cluster coordination ring dropped heartbeat signals at 14:22 UTC. Port 9092 is completely unreachable."
Metadata Flags: {'system': 'MeshQuery', 'category': 'Infrastructure', 'severity': 'Critical', 'tags': ['meshquery', 'heartbeat', 'unreachable']}

